# ETL Notebook 1: Trips Data Ingestion
Local pandas version for Windows development.
Reads raw trips data and records metrics.

## Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import json
from pathlib import Path

## Configuration

In [ ]:
# Configuration - Set these paths based on your local setup
DATA_PATH = "./data"  # Path to input data
ARTIFACT_PATH = "./artifacts"  # Path to save outputs

# File paths
SOURCE_FILE = os.path.join(DATA_PATH, "trips_raw.parquet")  # Input raw trips data
RAW_LAYER_FILE = os.path.join(ARTIFACT_PATH, "trips_raw.parquet")  # Output raw table
METRICS_FILE = os.path.join(ARTIFACT_PATH, "metrics.parquet")  # Metrics table

# Parameters
RUN_DATE = "2025-12-15"  # Date for this run

# Create artifact directory if it doesn't exist
os.makedirs(ARTIFACT_PATH, exist_ok=True)

print(f"Configuration:")
print(f"  Source file: {SOURCE_FILE}")
print(f"  Raw layer output: {RAW_LAYER_FILE}")
print(f"  Metrics output: {METRICS_FILE}")
print(f"  Run date: {RUN_DATE}")

## Step 1: Start Ingestion Timer

In [ ]:
ingestion_start_ts = datetime.utcnow()
print(f"Ingestion started at: {ingestion_start_ts}")

## Step 2: Load Raw Data

In [ ]:
# Read raw trips data
raw_trips_df = pd.read_parquet(SOURCE_FILE)

# Filter by run_date if date column exists
if 'date' in raw_trips_df.columns:
    raw_trips_df['date'] = pd.to_datetime(raw_trips_df['date'])
    run_date_dt = pd.to_datetime(RUN_DATE)
    raw_trips_df = raw_trips_df[raw_trips_df['date'].dt.date == run_date_dt.date()]

print(f"Loaded {len(raw_trips_df)} records")
print(f"Columns: {raw_trips_df.columns.tolist()}")

## Step 3: Calculate Ingestion Metrics

In [ ]:
# Basic record count
record_count = len(raw_trips_df)

# Null counts per column
null_counts = raw_trips_df.isnull().sum().to_dict()

# Min and max timestamps
min_start = raw_trips_df['start'].min() if 'start' in raw_trips_df.columns else None
max_end = raw_trips_df['end'].max() if 'end' in raw_trips_df.columns else None

print(f"Record count: {record_count}")
print(f"Null counts: {null_counts}")
print(f"Time range: {min_start} to {max_end}")

## Step 4: Calculate Distribution Statistics

In [ ]:
# Calculate distribution metrics for causal analysis
distribution_stats = {}

numeric_cols = {
    'distance': 'distance',
    'avg_speed': 'avg_speed',
    'fuel_consumption': 'fuel_consumption',
    'gps_coverage': 'gps_coverage'
}

for col in numeric_cols.values():
    if col in raw_trips_df.columns:
        distribution_stats[f"{col}_mean"] = raw_trips_df[col].mean()
        distribution_stats[f"{col}_std"] = raw_trips_df[col].std()

# Unique units
if 'unit_id' in raw_trips_df.columns:
    distribution_stats['unique_units'] = raw_trips_df['unit_id'].nunique()

# Poor GPS coverage count
if 'gps_coverage' in raw_trips_df.columns:
    distribution_stats['poor_gps_coverage_count'] = (raw_trips_df['gps_coverage'] < 0.8).sum()

# Temporal coverage (hours)
if min_start is not None and max_end is not None:
    temporal_coverage_hours = (max_end - min_start).total_seconds() / 3600 if hasattr(max_end, 'total_seconds') else (max_end - min_start) / 3600
    distribution_stats['temporal_coverage_hours'] = temporal_coverage_hours

print(f"Distribution stats: {distribution_stats}")

## Step 5: Save Raw Table

In [ ]:
# Save raw trips to parquet
raw_trips_df.to_parquet(RAW_LAYER_FILE, index=False)
print(f"✓ Raw trips saved to: {RAW_LAYER_FILE}")
print(f"  Shape: {raw_trips_df.shape}")

## Step 6: End Ingestion Timer

In [ ]:
ingestion_end_ts = datetime.utcnow()
ingestion_duration_sec = (ingestion_end_ts - ingestion_start_ts).total_seconds()
print(f"Ingestion completed at: {ingestion_end_ts}")
print(f"Duration: {ingestion_duration_sec:.2f} seconds")

## Step 7: Create Metrics Records

In [ ]:
# Create metrics dataframe
metrics_rows = []

run_date_obj = pd.to_datetime(RUN_DATE).date()
created_at_ts = datetime.utcnow()

# Add basic metrics
metrics_rows.append({
    'date': run_date_obj,
    'pipeline_stage': 'raw',
    'metric_name': 'raw_input_record_count',
    'metric_value': float(record_count),
    'created_at': created_at_ts
})

metrics_rows.append({
    'date': run_date_obj,
    'pipeline_stage': 'raw',
    'metric_name': 'raw_ingestion_duration_sec',
    'metric_value': float(ingestion_duration_sec),
    'created_at': created_at_ts
})

# Add distribution metrics
for metric_name, metric_value in distribution_stats.items():
    if metric_value is not None and not pd.isna(metric_value):
        metrics_rows.append({
            'date': run_date_obj,
            'pipeline_stage': 'raw',
            'metric_name': f"raw_{metric_name}",
            'metric_value': float(metric_value),
            'created_at': created_at_ts
        })

# Add null count metrics
for col_name, null_count in null_counts.items():
    metrics_rows.append({
        'date': run_date_obj,
        'pipeline_stage': 'raw',
        'metric_name': f"raw_null_count_{col_name}",
        'metric_value': float(null_count),
        'created_at': created_at_ts
    })

metrics_df = pd.DataFrame(metrics_rows)
print(f"Created {len(metrics_df)} metric records")
print(metrics_df.head(10))

## Step 8: Save Metrics

In [ ]:
# Append metrics to existing file or create new
if os.path.exists(METRICS_FILE):
    existing_metrics = pd.read_parquet(METRICS_FILE)
    metrics_df = pd.concat([existing_metrics, metrics_df], ignore_index=True)

metrics_df.to_parquet(METRICS_FILE, index=False)
print(f"✓ Metrics saved to: {METRICS_FILE}")
print(f"  Total metrics in file: {len(metrics_df)}")

## Step 9: Summary

In [ ]:
print("\n" + "="*60)
print("RAW INGESTION COMPLETED")
print("="*60)
print(f"Date: {RUN_DATE}")
print(f"Records ingested: {record_count}")
print(f"Ingestion duration: {ingestion_duration_sec:.2f} seconds")
print(f"Output file: {RAW_LAYER_FILE}")
print(f"Metrics recorded: {len(metrics_df)}")
print("="*60)